# Project 2 — E-commerce Funnel and A/B Test

Analyze an illustrative randomized product experiment. Quantify funnel loss, compare conversion rates, and communicate a decision with uncertainty. The user-level data are synthetic.

## Load data and inspect experiment design

The dataset is user-level: one row per eligible experiment user, with nested funnel indicators. Confirm each user appears once and that assignment is reasonably balanced before analyzing outcomes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path  # Find the project data directory.
DATA_DIR = Path("data") if Path("data").exists() else Path("../data")  # Support common notebook launch locations.
df = pd.read_csv(DATA_DIR / "experiment_users.csv", parse_dates=["experiment_date"])  # Load the experiment dataset.
print("Shape:", df.shape)  # Display the dataset dimensions.
print(df.head())  # Preview assigned variants and outcomes.
print("Duplicate user IDs:", df["user_id"].duplicated().sum())  # Confirm one row per user.
print("\nUsers per variant:\n", df["variant"].value_counts())  # Check the sample ratio before interpreting outcomes.
print("\nMissing values:\n", df.isna().sum())  # Inspect data completeness.

## Analyze funnel conversion

Compute step counts and rates using the same eligible-user denominator. Step-to-step rates use the previous step; overall conversion uses all eligible users. Keep the two definitions distinct.

In [ ]:
funnel_steps = ["product_view", "add_to_cart", "checkout_started", "purchase"]  # Define the funnel stage columns.
funnel_rows = []  # Create a list to store stage summaries.
previous_count = len(df)  # Set the first previous stage to all eligible users.
for stage in funnel_steps:  # Loop through each funnel stage.
    count = int(df[stage].sum())  # Count users who reached the stage.
    funnel_rows.append({"stage": stage, "users": count, "rate_from_previous": count / previous_count, "rate_from_all_users": count / len(df)})  # Store rates and counts.
    previous_count = count  # Use current stage count as denominator for the next stage.
funnel_summary = pd.DataFrame(funnel_rows)  # Convert the stage results into a DataFrame.
print(funnel_summary.to_string(index=False, formatters={"rate_from_previous": lambda x: f"{x:.1%}", "rate_from_all_users": lambda x: f"{x:.1%}"}))  # Display the funnel report.

## Compare conversion by variant

Compare variant conversion rates with a preselected primary metric. Report the absolute percentage-point difference and relative lift; use uncertainty and test-design checks before recommending a rollout.

In [ ]:
group_summary = df.groupby("variant").agg(  # Summarize outcomes for each experimental variant.
    users=("user_id", "nunique"),  # Count unique assigned users.
    conversions=("purchase", "sum"),  # Count users who purchased.
    revenue=("revenue", "sum"),  # Sum observed revenue.
)  # Finish group-level aggregation.
group_summary["conversion_rate"] = group_summary["conversions"] / group_summary["users"]  # Calculate conversion rate.
group_summary["revenue_per_user"] = group_summary["revenue"] / group_summary["users"]  # Calculate revenue per assigned user.
print(group_summary.round(4))  # Display conversion and revenue metrics by variant.
rate_a = group_summary.loc["A", "conversion_rate"]  # Read control conversion rate.
rate_b = group_summary.loc["B", "conversion_rate"]  # Read treatment conversion rate.
print(f"Absolute lift B-A: {rate_b-rate_a:.2%} percentage points")  # Display the absolute difference.
print(f"Relative lift B vs A: {(rate_b/rate_a-1):.1%}")  # Display relative lift compared with control.

## Test the conversion-rate difference

This lesson uses a two-proportion z-test as a large-sample approximation. It assumes independent users and sufficient expected counts. The p-value is not the probability the treatment is “true,” and it does not quantify business importance.

In [ ]:
a = df[df["variant"] == "A"]  # Select control users.
b = df[df["variant"] == "B"]  # Select treatment users.
x_a, n_a = int(a["purchase"].sum()), len(a)  # Count control conversions and users.
x_b, n_b = int(b["purchase"].sum()), len(b)  # Count treatment conversions and users.
pooled_rate = (x_a + x_b) / (n_a + n_b)  # Estimate the shared rate under the null hypothesis.
se_null = np.sqrt(pooled_rate * (1-pooled_rate) * (1/n_a + 1/n_b))  # Calculate the null standard error.
z_stat = ((x_b/n_b) - (x_a/n_a)) / se_null  # Standardize the observed rate difference.
p_value = 2 * stats.norm.sf(abs(z_stat))  # Calculate a two-sided p-value.
se_difference = np.sqrt((x_a/n_a)*(1-x_a/n_a)/n_a + (x_b/n_b)*(1-x_b/n_b)/n_b)  # Estimate the unpooled standard error for the difference.
difference = (x_b/n_b) - (x_a/n_a)  # Calculate treatment minus control.
ci = (difference - 1.96*se_difference, difference + 1.96*se_difference)  # Calculate an approximate 95% confidence interval.
print("z statistic:", round(z_stat, 3))  # Display the test statistic.
print("Two-sided p-value:", round(p_value, 4))  # Display the test result.
print("Approximate 95% CI:", tuple(round(v, 4) for v in ci))  # Display uncertainty around the absolute difference.
print("Also verify randomization, sample ratio, exposure rules, and guardrail metrics.")  # Remind the analyst to check the experiment design.

## Explore segments and prepare a decision

Segment results can suggest where an intervention may help, but multiple comparisons make false discoveries more likely. Treat subgroup findings as exploratory unless the test plan specified them in advance.

In [ ]:
segment_results = df.groupby(["device", "variant"]).agg(  # Compare variants within each device group.
    users=("user_id", "nunique"),  # Count assigned users.
    conversions=("purchase", "sum"),  # Count conversions.
    revenue=("revenue", "sum"),  # Sum revenue.
)  # Finish segment aggregation.
segment_results["conversion_rate"] = segment_results["conversions"] / segment_results["users"]  # Calculate conversion rate within each device-variant segment.
segment_results["revenue_per_user"] = segment_results["revenue"] / segment_results["users"]  # Calculate revenue per assigned user.
print(segment_results.round(4))  # Display segment outcomes.
print("Decision template: launch / do not launch / extend test, with effect size, uncertainty, guardrails, and expected business value.")  # State the required decision elements.